In [45]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np


class Tensor:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = np.zeros(data.shape)
        self._backward = lambda : None
        self.shape = data.shape
        self.size = data.size

    def __add__(self, other):
        out = Tensor(self.data + other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad, self.grad)
            other.grad += unbroadcast(out.grad, other.grad)
        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Tensor(self.data * other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad * other.data, self.grad)
            other.grad += unbroadcast(out.grad * self.data, other.grad)
        out._backward = _backward

        return out

    def __neg__(self):
        out = Tensor(-self.data, (self,))
            
        def _backward():
            self.grad -= out.grad
        out._backward = _backward

        return out

    def __sub__(self, other):
        return self + -other

    def __pow__(self, other):
        out = Tensor(self.data ** other, (self,))

        def _backward():
            self_grad = other * self.data ** (other - 1) * out.grad
            self.grad += unbroadcast(self_grad, self.grad)
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * (other ** -1)

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad @ other.data.T, self.grad)
            other.grad += unbroadcast(self.data.T @ out.grad, other.grad)
        out._backward = _backward

        return out

    def sum(self, dim=None):
        out = Tensor(self.data.sum(dim), (self,))

        def _backward():
            self.grad += out.grad if dim is None else np.expand_dims(out.grad, dim) 
        out._backward = _backward
 
        return out

    def mean(self, dim=None):
        out = Tensor(self.data.mean(dim), (self,))

        def _backward():
            if dim is None:
                self.grad += out.grad / self.grad.size
            else:
                self.grad += np.expand_dims(out.grad, dim) / self.shape[dim] 
        out._backward = _backward

        return out

    def exp(self):
        out = Tensor(np.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def log(self):
        out = Tensor(np.log(self.data), (self,))
        
        def _backward():
            self.grad += 1 / self.data * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,))

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def softmax(self, dim=-1):
        normal_sum = np.expand_dims(np.exp(self.data).sum(dim), dim)
        out = Tensor(np.exp(self.data) / normal_sum, (self,))

        def _backward():
            self.grad += out.data * (out.grad - np.sum(out.data * out.grad, axis=dim, keepdims=True))
        out._backward = _backward

        return out

    def cross_entropy(self, y):
        m = self.data.max(-1, keepdims=True)
        lse = m + np.log(np.sum(np.exp(self.data - m), -1, keepdims=True))
        out = Tensor((lse - np.sum(self.data * y, -1, keepdims=True)).mean(), (self,))

        def _backward():
            self.grad += (np.exp(self.data - lse) - y) / self.shape[0] * out.grad
        out._backward = _backward

        return out

    def backward(self, R=None):
        topo = []
        visited = set()
        
        def topo_sort(node):
            if node in visited:
                return
            visited.add(node)
            for child in node.children:
                topo_sort(child)
            topo.append(node)

        self.grad = np.ones(self.shape) if R is None else R
        topo_sort(self)
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
            return f'{self.data}'


def unbroadcast(input, target):
    target_shape = target.shape

    while len(target.shape) < len(input.shape):
        target = np.expand_dims(target, axis=0)
    for axis, dim in enumerate(target.shape):
        if dim == 1:
            input = np.sum(input, axis=axis, keepdims=True)

    return input.reshape(target_shape)


def gradcheck(f, inputs):
    inputs = [Tensor(input.data.astype(np.float64)) for input in inputs]
    out = f(*inputs)
    R = np.random.randn(*out.shape)
    out.backward(R)

    EPS = 1e-6
    for i in range(len(inputs)):
        inputs_pos = inputs.copy()
        inputs_neg = inputs.copy()
        for idx in np.ndindex(inputs[i].shape):
            input_pos = inputs[i].data.copy()
            input_neg = inputs[i].data.copy()
            input_pos[idx] += EPS
            input_neg[idx] -= EPS
            inputs_pos[i] = Tensor(input_pos)
            inputs_neg[i] = Tensor(input_neg)
            l_pos = np.sum(f(*inputs_pos).data * R)
            l_neg = np.sum(f(*inputs_neg).data * R)
            analytical = inputs[i].grad[idx]
            numerical = (l_pos - l_neg) / (EPS * 2)
            assert np.allclose(analytical, numerical, rtol=1e-5, atol=1e-6)


class Layer:

    def __init__(self, nin, nout):
        self.weights = Tensor(np.random.randn(nin, nout) / nin ** 0.5)
        self.bias = Tensor(np.random.randn(nout) / nin ** 0.5)

    def __call__(self, x):
        return x @ self.weights + self.bias


class MLP:

    def __init__(self):
        self.fc1 = Layer(784, 512)
        self.fc2 = Layer(512, 64)
        self.out = Layer(64, 10)

        self.layers = (self.fc1, self.fc2, self.out)

    def __call__(self, x):
        x = self.fc1(x).relu()
        x = self.fc2(x).relu()
        x = self.out(x)

        return x

    def parameters(self):
        return [param for l in self.layers for param in (l.weights, l.bias)]

    def step(self, learning_rate):
        for param in self.parameters():
            param.data -= learning_rate * param.grad

    def zero_grad(self):
        for param in self.parameters():
                param.grad *= 0

In [52]:
BATCH_SIZE = 100

train_data = datasets.MNIST(root='../data',
                            download=True,
                            train=True,
                            transform=transforms.Lambda(lambda x : np.array(x).flatten()))

test_data = datasets.MNIST(root='../data',
                           download=True,
                           train=False,
                           transform=transforms.Lambda(lambda x : np.array(x).flatten()))

x_train, y_train = [], []
x_batch, y_batch = [], []
x_val, y_val = [], []

for i, (x, y) in enumerate(train_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        if i <= 50000:
            x_train.append(Tensor(np.array(x_batch)))
            y_train.append(Tensor(np.array(y_batch)))
        else:
            x_val.append(Tensor(np.array(x_batch)))
            y_val.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []

    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

x_test, y_test = [], []
x_batch, y_batch = [], []

for i, (x, y) in enumerate(test_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        x_test.append(Tensor(np.array(x_batch)))
        y_test.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []
    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

len(x_train), len(y_train), len(x_val), len(y_val), len(x_test), len(y_test)

(500, 500, 100, 100, 100, 100)

In [53]:
EPOCHS = 100
LEARNING_RATE = 1e-2

model = MLP()

for epoch in range(1, EPOCHS + 1):
    for x, y in zip(x_train, y_train):
        y_pred = model(x)
        loss = y_pred.cross_entropy(y.data)
        model.zero_grad()
        loss.backward()
        model.step(LEARNING_RATE)

    if epoch % 10 == 0:
        print(f'epoch: {epoch}, loss: {loss.data:.4f}')

epoch: 10, loss: 0.2255
epoch: 20, loss: 0.1305
epoch: 30, loss: 0.0830
epoch: 40, loss: 0.0590
epoch: 50, loss: 0.0464
epoch: 60, loss: 0.0396
epoch: 70, loss: 0.0349
epoch: 80, loss: 0.0298
epoch: 90, loss: 0.0251
epoch: 100, loss: 0.0202


In [54]:
total = len(test_data)
correct = 0
for x, y in zip(x_test, y_test):
    y_pred = model(x)
    correct += np.sum(np.argmax(y.data, -1) == np.argmax(y_pred.data, -1))

print(f'loss: {loss.data:.4f}, accuracy: {correct}/{total} = {correct/total*100:.4f}%')

loss: 0.0202, accuracy: 9775/10000 = 97.7500%
